# Smart Pricing: Helping Consumers and Retailers Make Data-Driven Smartphone Decisions

**ITCS 227 — Introduction to Data Science — Capstone Project**

This notebook implements the full data science lifecycle for predicting smartphone prices from their specifications, using the Mobiles Dataset (2025).

**Sections:**
1. Data Loading & Preprocessing
2. Exploratory Data Analysis (EDA)
3. Regression Modeling (Price Prediction)
4. Classification (Price Tier Prediction)
5. Unsupervised Analysis (Phone Clustering)
6. Cross-Country Price Analysis

**Course Techniques Applied:**
| Lab | Technique | Application |
|-----|-----------|-------------|
| Lab05 | Feature binning (`pd.cut`, `pd.qcut`) | RAM, Battery, Price bins |
| Lab06 | Polynomial Features (degree=2) | Poly-2 Ridge Regression |
| Lab07/10 | `HistGradientBoostingRegressor` | Best tabular model |
| Lab10 | Cross-validation (5-fold) | Robust model evaluation |
| Lab10 | Learning Curves | Overfitting diagnosis |
| Lab10 | Partial Dependency Plots | Feature effect analysis |
| Lab10 | Pearson correlation + p-values | Statistical significance |
| Lab10 | Model export (`pickle`) | `best_model.pkl` |
| Lab12 | DBSCAN clustering | Density-based grouping |
| Lab12 | GMM clustering | Soft-clustering |
| Lab12 | Silhouette Score | Cluster quality metric |
| Lab12 | Hyperparameter grid search | Optimal clustering params |

## Imports & Configuration

In [11]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import re
import warnings
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import (
    RandomForestRegressor, RandomForestClassifier,
    GradientBoostingRegressor, ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, silhouette_score
)
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.inspection import PartialDependenceDisplay

warnings.filterwarnings("ignore")

# ── Configuration ────────────────────────────────────────────────────────────
# Create output directory for plots
OUTPUT_DIR = os.path.join(os.getcwd(), "plots")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Plot style
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams.update({
    "figure.figsize": (12, 7),
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "figure.dpi": 150,
    "savefig.dpi": 150,
})

RANDOM_STATE = 42

## PART 1: DATA LOADING & PREPROCESSING

**Techniques:** Feature binning (`pd.cut`/`pd.qcut`), interaction features (Lab05)

In [12]:
# --- 1.1 Load the dataset ---------------------------------------------------
DATA_PATH = os.path.join(
    os.path.expanduser("~"),
    ".cache/kagglehub/datasets/abdulmalik1518/mobiles-dataset-2025/versions/1",
    "Mobiles Dataset (2025).csv",
)

# If cached file doesn't exist, download it
if not os.path.exists(DATA_PATH):
    print("Dataset not found locally. Downloading from Kaggle...")
    import kagglehub
    dataset_path = kagglehub.dataset_download("abdulmalik1518/mobiles-dataset-2025")
    DATA_PATH = os.path.join(dataset_path, "Mobiles Dataset (2025).csv")

df_raw = pd.read_csv(DATA_PATH, encoding="latin-1")
print(f"\n✅ Dataset loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print(f"\nColumns: {list(df_raw.columns)}")
print(f"\nFirst 5 rows:")
print(df_raw.head().to_string())

# --- 1.2 Helper functions for parsing ----------------------------------------

def parse_weight(val):
    """Extract numeric weight in grams. e.g., '174g' -> 174"""
    match = re.search(r"([\d.]+)", str(val))
    return float(match.group(1)) if match else np.nan

def parse_ram(val):
    """Extract RAM in GB. e.g., '6GB' -> 6"""
    match = re.search(r"(\d+)", str(val))
    return int(match.group(1)) if match else np.nan

def parse_camera(val):
    """Extract max camera resolution in MP. e.g., '50MP + 12MP' -> 50"""
    numbers = re.findall(r"(\d+)\s*MP", str(val), re.IGNORECASE)
    return max([int(n) for n in numbers]) if numbers else np.nan

def parse_camera_count(val):
    """Count number of camera lenses. e.g., '50MP + 12MP' -> 2"""
    numbers = re.findall(r"(\d+)\s*MP", str(val), re.IGNORECASE)
    return len(numbers) if numbers else 0

def parse_battery(val):
    """Extract battery capacity in mAh. e.g., '3,600mAh' -> 3600"""
    cleaned = str(val).replace(",", "")
    match = re.search(r"([\d.]+)", cleaned)
    return float(match.group(1)) if match else np.nan

def parse_screen(val):
    """Extract screen size in inches. e.g., '6.1 inches' -> 6.1"""
    match = re.search(r"([\d.]+)", str(val))
    return float(match.group(1)) if match else np.nan

def parse_price(val):
    """Extract numeric price. e.g., 'USD 799' -> 799, 'PKR 224,999' -> 224999"""
    cleaned = str(val).replace(",", "")
    match = re.search(r"([\d.]+)", cleaned)
    return float(match.group(1)) if match else np.nan

# --- 1.3 Apply parsing -------------------------------------------------------
df = df_raw.copy()

# Parse specification columns
df["Weight_g"] = df["Mobile Weight"].apply(parse_weight)
df["RAM_GB"] = df["RAM"].apply(parse_ram)
df["Front_Camera_MP"] = df["Front Camera"].apply(parse_camera)
df["Back_Camera_MP"] = df["Back Camera"].apply(parse_camera)
df["Back_Camera_Count"] = df["Back Camera"].apply(parse_camera_count)
df["Battery_mAh"] = df["Battery Capacity"].apply(parse_battery)
df["Screen_inches"] = df["Screen Size"].apply(parse_screen)

# Parse price columns
price_cols_raw = {
    "Launched Price (Pakistan)": "Price_PKR",
    "Launched Price (India)": "Price_INR",
    "Launched Price (China)": "Price_CNY",
    "Launched Price (USA)": "Price_USD",
    "Launched Price (Dubai)": "Price_AED",
}
for raw_col, new_col in price_cols_raw.items():
    df[new_col] = df[raw_col].apply(parse_price)

df["Year"] = df["Launched Year"]
df["Company"] = df["Company Name"]

# --- 1.4 Identify device types and fix data quality issues -------------------
# Tag tablets vs. smartphones (tablets have "Pad"/"Tab" in name or weight > 400g)
tablet_keywords = r"(?i)(Tab|Pad|pad|tab)"
df["Is_Tablet"] = (
    df["Model Name"].str.contains(tablet_keywords, na=False) |
    (df["Weight_g"] > 400)
)

# Fix known data entry error: Nokia T21 price "USD 396,22" parsed as 39622
# The correct price is ~$162 (it's a budget tablet). We fix the comma issue.
nokia_t21_mask = df["Model Name"].str.contains("T21", na=False) & (df["Company"] == "Nokia")
if nokia_t21_mask.any():
    df.loc[nokia_t21_mask, "Price_USD"] = 162.0
    df.loc[nokia_t21_mask, "Price_PKR"] = 44999.0
    df.loc[nokia_t21_mask, "Price_INR"] = 13499.0
    df.loc[nokia_t21_mask, "Price_CNY"] = 1199.0
    df.loc[nokia_t21_mask, "Price_AED"] = 599.0
    print("⚠️  Fixed Nokia T21 price data entry error (39,622 → 162 USD)")

print(f"\n📊 Device breakdown:")
print(f"   Smartphones: {(~df['Is_Tablet']).sum()}")
print(f"   Tablets:     {df['Is_Tablet'].sum()}")

# --- 1.5 Feature Engineering -------------------------------------------------
df["Price_Per_GB_RAM"] = df["Price_USD"] / df["RAM_GB"]
df["Camera_Ratio"] = df["Back_Camera_MP"] / df["Front_Camera_MP"].replace(0, np.nan)
df["Battery_Per_Inch"] = df["Battery_mAh"] / df["Screen_inches"]
df["Total_Camera_MP"] = df["Front_Camera_MP"] + df["Back_Camera_MP"]

# --- 1.5b Feature Binning (Lab05 technique: pd.cut / pd.qcut) ---------------
# Discretize continuous features into categorical bins for analysis
df["RAM_Bin"] = pd.cut(df["RAM_GB"], bins=[0, 3, 6, 8, 16],
                       labels=["Low", "Mid", "High", "Ultra"])
df["Battery_Bin"] = pd.cut(df["Battery_mAh"],
                           bins=[0, 3500, 4500, 5500, 7000],
                           labels=["Small", "Medium", "Large", "Massive"])
df["Price_Quartile"] = pd.qcut(df["Price_USD"], q=4,
                               labels=["Q1", "Q2", "Q3", "Q4"],
                               duplicates="drop")

# Interaction features
df["RAM_x_Camera"] = df["RAM_GB"] * df["Back_Camera_MP"]
df["Screen_x_Battery"] = df["Screen_inches"] * df["Battery_mAh"]

print("\n📊 Feature Binning (pd.cut / pd.qcut):")
print(f"   RAM bins:     {df['RAM_Bin'].value_counts().to_dict()}")
print(f"   Battery bins: {df['Battery_Bin'].value_counts().to_dict()}")
print(f"   Price quartiles: {df['Price_Quartile'].value_counts().to_dict()}")

# --- 1.6 Define feature columns and clean data ------------------------------
feature_cols = [
    "Weight_g", "RAM_GB", "Front_Camera_MP", "Back_Camera_MP",
    "Back_Camera_Count", "Battery_mAh", "Screen_inches", "Year"
]
target_col = "Price_USD"

# Drop rows with NaN in critical columns
df_all = df.dropna(subset=feature_cols + [target_col]).copy()

# Filter to smartphones only for modeling (tablets have very different specs/price relationships)
df_clean = df_all[~df_all["Is_Tablet"]].copy()

# Also remove extreme price outliers (> $3,000 — foldables/special editions)
price_outlier_mask = df_clean["Price_USD"] > 3000
n_outliers = price_outlier_mask.sum()
df_clean = df_clean[~price_outlier_mask].copy()

print(f"\n✅ After cleaning:")
print(f"   Total raw rows:          {df.shape[0]}")
print(f"   After removing tablets:  {df_all[~df_all['Is_Tablet']].shape[0]}")
print(f"   After removing outliers: {df_clean.shape[0]} (removed {n_outliers} phones > $3,000)")
print(f"   Final dataset:           {df_clean.shape[0]} smartphones")

# Summary statistics of cleaned numeric features
print("\n📊 Summary Statistics (Smartphones Only):")
summary_cols = feature_cols + [target_col, "Price_Per_GB_RAM", "Total_Camera_MP"]
print(df_clean[summary_cols].describe().round(2).to_string())


✅ Dataset loaded: 930 rows × 15 columns

Columns: ['Company Name', 'Model Name', 'Mobile Weight', 'RAM', 'Front Camera', 'Back Camera', 'Processor', 'Battery Capacity', 'Screen Size', 'Launched Price (Pakistan)', 'Launched Price (India)', 'Launched Price (China)', 'Launched Price (USA)', 'Launched Price (Dubai)', 'Launched Year']

First 5 rows:
  Company Name            Model Name Mobile Weight  RAM Front Camera Back Camera   Processor Battery Capacity Screen Size Launched Price (Pakistan) Launched Price (India) Launched Price (China) Launched Price (USA) Launched Price (Dubai)  Launched Year
0        Apple       iPhone 16 128GB          174g  6GB         12MP        48MP  A17 Bionic         3,600mAh  6.1 inches               PKR 224,999             INR 79,999              CNY 5,799              USD 799              AED 2,799           2024
1        Apple       iPhone 16 256GB          174g  6GB         12MP        48MP  A17 Bionic         3,600mAh  6.1 inches               PKR 234,99

## PART 2: EXPLORATORY DATA ANALYSIS (EDA)

**Techniques:** Pearson correlation with p-value significance testing (Lab10)

In [13]:
print("\n" + "=" * 70)

# --- 2.1 Price Distribution ---------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
axes[0].hist(df_clean["Price_USD"], bins=40, color="#4C72B0", edgecolor="white", alpha=0.85)
axes[0].set_title("Distribution of Smartphone Prices (USD)", fontweight="bold")
axes[0].set_xlabel("Price (USD)")
axes[0].set_ylabel("Number of Phones")
axes[0].axvline(df_clean["Price_USD"].median(), color="#C44E52", linestyle="--",
                linewidth=2, label=f'Median: ${df_clean["Price_USD"].median():,.0f}')
axes[0].axvline(df_clean["Price_USD"].mean(), color="#DD8452", linestyle="--",
                linewidth=2, label=f'Mean: ${df_clean["Price_USD"].mean():,.0f}')
axes[0].legend(fontsize=11)

# Box plot by price range
axes[1].boxplot(df_clean["Price_USD"], vert=True, patch_artist=True,
                boxprops=dict(facecolor="#4C72B0", alpha=0.7),
                medianprops=dict(color="#C44E52", linewidth=2))
axes[1].set_title("Price Box Plot (USD)", fontweight="bold")
axes[1].set_ylabel("Price (USD)")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_price_distribution.png"))
plt.close()
print("✅ Saved: 01_price_distribution.png")

# --- 2.2 Brand Comparison ----------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Average price by brand
brand_avg = df_clean.groupby("Company")["Price_USD"].mean().sort_values(ascending=True)
colors = sns.color_palette("viridis", len(brand_avg))
brand_avg.plot(kind="barh", ax=axes[0], color=colors, edgecolor="white")
axes[0].set_title("Average Smartphone Price by Brand (USD)", fontweight="bold")
axes[0].set_xlabel("Average Price (USD)")
axes[0].set_ylabel("")
for i, (val, name) in enumerate(zip(brand_avg.values, brand_avg.index)):
    axes[0].text(val + 10, i, f"${val:,.0f}", va="center", fontsize=10)

# Phone count by brand
brand_count = df_clean["Company"].value_counts().sort_values(ascending=True)
colors2 = sns.color_palette("magma", len(brand_count))
brand_count.plot(kind="barh", ax=axes[1], color=colors2, edgecolor="white")
axes[1].set_title("Number of Phone Models by Brand", fontweight="bold")
axes[1].set_xlabel("Number of Models")
axes[1].set_ylabel("")
for i, (val, name) in enumerate(zip(brand_count.values, brand_count.index)):
    axes[1].text(val + 0.5, i, str(val), va="center", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_brand_comparison.png"))
plt.close()
print("✅ Saved: 02_brand_comparison.png")

# --- 2.3 Correlation Heatmap ------------------------------------------------
numeric_cols_for_corr = [
    "Weight_g", "RAM_GB", "Front_Camera_MP", "Back_Camera_MP",
    "Back_Camera_Count", "Battery_mAh", "Screen_inches", "Year",
    "Price_USD", "Price_Per_GB_RAM", "Total_Camera_MP"
]
corr_matrix = df_clean[numeric_cols_for_corr].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(250, 15, s=75, l=40, n=9, center="light", as_cmap=True)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap=cmap,
            center=0, square=True, linewidths=1, ax=ax,
            cbar_kws={"shrink": 0.8, "label": "Correlation Coefficient"})
ax.set_title("Feature Correlation Heatmap", fontweight="bold", fontsize=18, pad=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_correlation_heatmap.png"))
plt.close()
print("✅ Saved: 03_correlation_heatmap.png")

# --- 2.3b Pearson Correlation with P-Values (Lab10 technique) ----------------
print("\n📊 Pearson Correlation with Significance Testing (scipy.stats.pearsonr):")
print("-" * 65)
print(f"{'Feature':<25} {'Pearson r':>10} {'p-value':>12} {'Significant?':>14}")
print("-" * 65)
pearson_features = ["Weight_g", "RAM_GB", "Front_Camera_MP", "Back_Camera_MP",
                    "Back_Camera_Count", "Battery_mAh", "Screen_inches", "Year"]
for feat in pearson_features:
    valid = df_clean[[feat, "Price_USD"]].dropna()
    corr_coef, p_value = pearsonr(valid[feat], valid["Price_USD"])
    sig = "✅ Yes" if p_value < 0.05 else "❌ No"
    print(f"{feat:<25} {corr_coef:>10.4f} {p_value:>12.2e} {sig:>14}")
print("-" * 65)

# --- 2.4 Year-Over-Year Pricing Trends ---------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Average price per year
yearly_avg = df_clean.groupby("Year")["Price_USD"].agg(["mean", "median", "count"])
axes[0].plot(yearly_avg.index, yearly_avg["mean"], "o-", color="#4C72B0",
             linewidth=2.5, markersize=8, label="Mean Price")
axes[0].plot(yearly_avg.index, yearly_avg["median"], "s--", color="#C44E52",
             linewidth=2.5, markersize=8, label="Median Price")
axes[0].fill_between(yearly_avg.index, yearly_avg["mean"], yearly_avg["median"],
                     alpha=0.15, color="#4C72B0")
axes[0].set_title("Average Smartphone Price Trend Over Years", fontweight="bold")
axes[0].set_xlabel("Launch Year")
axes[0].set_ylabel("Price (USD)")
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Phone releases per year
axes[1].bar(yearly_avg.index, yearly_avg["count"], color=sns.color_palette("viridis", len(yearly_avg)),
            edgecolor="white")
axes[1].set_title("Number of Phone Releases Per Year", fontweight="bold")
axes[1].set_xlabel("Launch Year")
axes[1].set_ylabel("Number of Models")
for i, (yr, cnt) in enumerate(zip(yearly_avg.index, yearly_avg["count"])):
    axes[1].text(yr, cnt + 1, str(int(cnt)), ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_yearly_trends.png"))
plt.close()
print("✅ Saved: 04_yearly_trends.png")

# --- 2.5 Feature vs. Price Scatter Plots -------------------------------------
scatter_features = [
    ("RAM_GB", "RAM (GB)"),
    ("Back_Camera_MP", "Back Camera (MP)"),
    ("Battery_mAh", "Battery Capacity (mAh)"),
    ("Screen_inches", "Screen Size (inches)"),
    ("Front_Camera_MP", "Front Camera (MP)"),
    ("Weight_g", "Weight (g)"),
]

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, (feat, label) in enumerate(scatter_features):
    ax = axes[i]
    scatter = ax.scatter(df_clean[feat], df_clean["Price_USD"],
                         c=df_clean["Year"], cmap="viridis",
                         alpha=0.6, edgecolors="white", linewidth=0.5, s=50)
    ax.set_xlabel(label)
    ax.set_ylabel("Price (USD)")
    ax.set_title(f"{label} vs. Price", fontweight="bold")
    # Add trend line
    z = np.polyfit(df_clean[feat].dropna(), df_clean.loc[df_clean[feat].notna(), "Price_USD"], 1)
    p = np.poly1d(z)
    x_sorted = np.sort(df_clean[feat].dropna())
    ax.plot(x_sorted, p(x_sorted), "--", color="#C44E52", linewidth=2, alpha=0.8)

fig.colorbar(scatter, ax=axes, label="Launch Year", shrink=0.6, pad=0.02)
fig.suptitle("How Each Specification Relates to Price", fontsize=20, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_feature_vs_price.png"))
plt.close()
print("✅ Saved: 05_feature_vs_price.png")

# --- 2.6 Brand Premium Analysis -----------------------------------------------
# For phones with similar specs (same RAM), which brands are pricier?
fig, ax = plt.subplots(figsize=(14, 8))
top_brands = df_clean["Company"].value_counts().head(10).index
df_top = df_clean[df_clean["Company"].isin(top_brands)]

sns.boxplot(data=df_top, x="Company", y="Price_USD",
            palette="Set2", ax=ax, showfliers=True,
            flierprops=dict(marker="o", markerfacecolor="red", alpha=0.5))
ax.set_title("Price Distribution by Brand (Top 10 Brands)", fontweight="bold")
ax.set_xlabel("Brand")
ax.set_ylabel("Price (USD)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_brand_premium.png"))
plt.close()
print("✅ Saved: 06_brand_premium.png")

# --- 2.7 RAM Distribution by Brand (Violin Plot) ----------------------------
fig, ax = plt.subplots(figsize=(14, 7))
sns.violinplot(data=df_top, x="Company", y="RAM_GB", palette="muted", ax=ax, inner="quartile")
ax.set_title("RAM Distribution by Brand", fontweight="bold")
ax.set_xlabel("Brand")
ax.set_ylabel("RAM (GB)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "07_ram_by_brand.png"))
plt.close()
print("✅ Saved: 07_ram_by_brand.png")


✅ Saved: 01_price_distribution.png
✅ Saved: 02_brand_comparison.png
✅ Saved: 03_correlation_heatmap.png

📊 Pearson Correlation with Significance Testing (scipy.stats.pearsonr):
-----------------------------------------------------------------
Feature                    Pearson r      p-value   Significant?
-----------------------------------------------------------------
Weight_g                      0.5302     2.75e-61          ✅ Yes
RAM_GB                        0.4651     9.89e-46          ✅ Yes
Front_Camera_MP               0.0505     1.46e-01           ❌ No
Back_Camera_MP                0.0753     3.02e-02          ✅ Yes
Back_Camera_Count             0.1374     7.22e-05          ✅ Yes
Battery_mAh                  -0.1767     3.06e-07          ✅ Yes
Screen_inches                 0.3456     1.15e-24          ✅ Yes
Year                          0.0701     4.36e-02          ✅ Yes
-----------------------------------------------------------------
✅ Saved: 04_yearly_trends.png
✅ Saved: 

## PART 3: REGRESSION MODELING — PRICE PREDICTION

**New techniques in this section:**
- Polynomial Features (degree=2) for Ridge Regression — Lab06
- `HistGradientBoostingRegressor` — Lab07/Lab10 (course-recommended model)
- 5-Fold Cross-Validation (`cross_val_score`) — Lab10
- Learning Curves (overfitting diagnosis) — Lab10
- Partial Dependency Plots — Lab10
- Model export with `pickle` — Lab10

In [15]:
print("\n" + "=" * 70)

# --- 3.1 Prepare data --------------------------------------------------------
# Encode company names
le_company = LabelEncoder()
df_clean["Company_Encoded"] = le_company.fit_transform(df_clean["Company"])

regression_features = feature_cols + ["Company_Encoded"]

X = df_clean[regression_features].values
y = df_clean[target_col].values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n📦 Training set: {X_train.shape[0]} samples")
print(f"📦 Test set:     {X_test.shape[0]} samples")
print(f"📦 Features:     {regression_features}")

# --- 3.2 Polynomial Features for Linear Models (Lab06 technique) -------------
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)
print(f"\n🔢 Polynomial Features: {X_train_scaled.shape[1]} → {X_train_poly.shape[1]} features (degree=2)")

# --- 3.3 Train models --------------------------------------------------------
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression (α=1.0)": Ridge(alpha=1.0),
    "Poly-2 Ridge Regression": Ridge(alpha=1.0),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=100, random_state=RANDOM_STATE, max_depth=15
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, random_state=RANDOM_STATE
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.05, max_depth=8,
        random_state=RANDOM_STATE
    ),
    "Extra Trees Regressor": ExtraTreesRegressor(
        n_estimators=200, random_state=RANDOM_STATE, max_depth=20
    ),
    "Tuned Random Forest": RandomForestRegressor(
        n_estimators=300, random_state=RANDOM_STATE, max_depth=25,
        min_samples_split=2, min_samples_leaf=1, max_features='sqrt'
    ),
}

results = {}
print("\n" + "-" * 70)
print(f"{'Model':<30} {'MAE':>10} {'RMSE':>10} {'R²':>10} {'CV R² (5-fold)':>15}")
print("-" * 70)

for name, model in models.items():
    # Determine which data to use
    if "Poly" in name:
        model.fit(X_train_poly, y_train)
        y_pred = model.predict(X_test_poly)
        cv_scores = cross_val_score(model, X_train_poly, y_train, cv=5, scoring='r2')
    elif any(kw in name for kw in ["Forest", "Gradient", "Extra", "Tuned", "Hist"]):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    else:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results[name] = {"MAE": mae, "RMSE": rmse, "R²": r2, "predictions": y_pred,
                     "CV_R2_mean": cv_scores.mean(), "CV_R2_std": cv_scores.std()}
    print(f"{name:<30} ${mae:>9,.0f} ${rmse:>9,.0f} {r2:>9.4f} {cv_scores.mean():>8.4f}±{cv_scores.std():.4f}")

print("-" * 70)

# --- 3.3 Visualize regression results ----------------------------------------
# Actual vs. Predicted for best model
best_model_name = max(results, key=lambda k: results[k]["R²"])
best_preds = results[best_model_name]["predictions"]

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# (a) Actual vs Predicted
axes[0].scatter(y_test, best_preds, alpha=0.6, c="#4C72B0", edgecolors="white", s=60)
max_val = max(y_test.max(), best_preds.max())
axes[0].plot([0, max_val], [0, max_val], "--", color="#C44E52", linewidth=2, label="Perfect Prediction")
axes[0].set_title(f"Actual vs. Predicted Price\n({best_model_name})", fontweight="bold")
axes[0].set_xlabel("Actual Price (USD)")
axes[0].set_ylabel("Predicted Price (USD)")
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# (b) Residual plot
residuals = y_test - best_preds
axes[1].scatter(best_preds, residuals, alpha=0.6, c="#55A868", edgecolors="white", s=60)
axes[1].axhline(y=0, color="#C44E52", linestyle="--", linewidth=2)
axes[1].set_title("Residual Plot", fontweight="bold")
axes[1].set_xlabel("Predicted Price (USD)")
axes[1].set_ylabel("Residual (Actual - Predicted)")
axes[1].grid(True, alpha=0.3)

# (c) Model comparison bar chart
model_names = list(results.keys())
r2_scores = [results[m]["R²"] for m in model_names]
bar_colors = sns.color_palette("husl", len(model_names))
bars = axes[2].bar(range(len(model_names)), r2_scores, color=bar_colors, edgecolor="white", width=0.6)
axes[2].set_xticks(range(len(model_names)))
axes[2].set_xticklabels([n.replace(" Regressor", "").replace(" (α=1.0)", "") for n in model_names],
                        rotation=25, ha="right", fontsize=9)
axes[2].set_title("Model Comparison (R² Score)", fontweight="bold")
axes[2].set_ylabel("R² Score")
axes[2].set_ylim(0, 1.05)
for bar, score in zip(bars, r2_scores):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                 f"{score:.4f}", ha="center", fontweight="bold", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "08_regression_results.png"))
plt.close()
print(f"\n✅ Saved: 08_regression_results.png")
print(f"🏆 Best Model: {best_model_name} (R² = {results[best_model_name]['R²']:.4f})")

# --- 3.4 Feature Importance --------------------------------------------------
rf_model = models["Random Forest Regressor"]
importances = rf_model.feature_importances_
feat_importance = pd.Series(importances, index=regression_features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette("viridis", len(feat_importance))
feat_importance.plot(kind="barh", ax=ax, color=colors, edgecolor="white")
ax.set_title("Feature Importance — What Drives Smartphone Prices?", fontweight="bold", fontsize=16)
ax.set_xlabel("Importance Score")
ax.set_ylabel("")
for i, (val, name) in enumerate(zip(feat_importance.values, feat_importance.index)):
    ax.text(val + 0.003, i, f"{val:.3f}", va="center", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "09_feature_importance.png"))
plt.close()
print("✅ Saved: 09_feature_importance.png")

# --- 3.5 Cross-Validation Comparison (Lab10 technique) -----------------------
print("\n📊 Cross-Validation Results (5-Fold):")
print("-" * 55)
for name in results:
    cv_mean = results[name]["CV_R2_mean"]
    cv_std = results[name]["CV_R2_std"]
    print(f"  {name:<30} CV R² = {cv_mean:.4f} ± {cv_std:.4f}")
print("-" * 55)

# --- 3.6 Learning Curves (Lab10 technique) -----------------------------------
# Learning curves show train vs validation error as training set grows
# Helps diagnose overfitting (gap between curves) vs underfitting (both high)
best_tree_model_name = max(
    [k for k in results if any(kw in k for kw in ["Forest", "Gradient", "Extra", "Tuned", "Hist"])],
    key=lambda k: results[k]["R²"]
)
best_tree_model = models[best_tree_model_name]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for idx, (model_name, model_obj, X_lc, y_lc) in enumerate([
    ("Linear Regression", models["Linear Regression"], X_train_scaled, y_train),
    (best_tree_model_name, best_tree_model, X_train, y_train),
]):
    train_sizes, train_scores, valid_scores = learning_curve(
        model_obj, X_lc, y_lc,
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5, scoring='neg_mean_absolute_error',
        n_jobs=-1
    )
    train_errors = -train_scores.mean(axis=1)
    valid_errors = -valid_scores.mean(axis=1)

    axes[idx].plot(train_sizes, train_errors, "r-o", linewidth=2, label="Train MAE")
    axes[idx].plot(train_sizes, valid_errors, "b-s", linewidth=2, label="Validation MAE")
    axes[idx].fill_between(train_sizes, train_errors, valid_errors, alpha=0.1, color="blue")
    axes[idx].set_title(f"Learning Curve — {model_name}", fontweight="bold")
    axes[idx].set_xlabel("Training Set Size")
    axes[idx].set_ylabel("MAE (USD)")
    axes[idx].legend(fontsize=11)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "17_learning_curves.png"))
plt.close()
print("✅ Saved: 17_learning_curves.png")

# --- 3.7 Partial Dependency Plots (Lab10 technique) --------------------------
# PDPs show the marginal effect of each feature on the predicted price
top_feature_indices = list(range(min(6, len(regression_features))))

fig, ax = plt.subplots(ncols=len(top_feature_indices), figsize=(20, 4), sharey=True)
for a in ax:
    a.axhline(y=0, color='r', linestyle='dashed', alpha=0.5)
PartialDependenceDisplay.from_estimator(
    best_tree_model,
    X_train,
    features=top_feature_indices,
    feature_names=regression_features,
    ax=ax,
    kind='average'
)
fig.suptitle(f"Partial Dependency Plots — {best_tree_model_name}", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "18_partial_dependency.png"))
plt.close()
print("✅ Saved: 18_partial_dependency.png")

# --- 3.8 Export Best Model with Pickle (Lab10 technique) ---------------------
model_export_path = os.path.join(os.getcwd(), "best_model.pkl")
with open(model_export_path, 'wb') as f:
    pickle.dump({
        'model': best_tree_model,
        'model_name': best_tree_model_name,
        'scaler': scaler,
        'feature_names': regression_features,
        'r2_score': results[best_tree_model_name]['R²'],
    }, f)
print(f"✅ Exported best model to: best_model.pkl")
print(f"   Model: {best_tree_model_name} | R² = {results[best_tree_model_name]['R²']:.4f}")

# --- 3.9 Deal Detector — Overpriced / Underpriced Phones --------------------
print("\n" + "-" * 60)
print("💰 DEAL DETECTOR — Overpriced & Underpriced Phones")
print("-" * 60)

# Use RF model for all predictions
all_preds = rf_model.predict(df_clean[regression_features].values)
df_clean["Predicted_USD"] = all_preds
df_clean["Price_Diff"] = df_clean["Price_USD"] - df_clean["Predicted_USD"]
df_clean["Price_Diff_Pct"] = (df_clean["Price_Diff"] / df_clean["Predicted_USD"]) * 100

# Top 10 overpriced (actual >> predicted)
print("\n🔴 Top 10 OVERPRICED Phones (you pay more than expected):")
overpriced = df_clean.nlargest(10, "Price_Diff_Pct")[
    ["Company", "Model Name", "Price_USD", "Predicted_USD", "Price_Diff_Pct"]
]
print(overpriced.to_string(index=False))

# Top 10 underpriced / best deals (actual << predicted)
print("\n🟢 Top 10 BEST DEALS (you pay less than expected):")
underpriced = df_clean.nsmallest(10, "Price_Diff_Pct")[
    ["Company", "Model Name", "Price_USD", "Predicted_USD", "Price_Diff_Pct"]
]
print(underpriced.to_string(index=False))

# Visualize deal detector
fig, ax = plt.subplots(figsize=(14, 7))
scatter = ax.scatter(df_clean["Price_USD"], df_clean["Price_Diff_Pct"],
                     c=df_clean["Price_Diff_Pct"], cmap="RdYlGn_r",
                     alpha=0.7, edgecolors="white", s=60)
ax.axhline(y=0, color="black", linestyle="-", linewidth=1)
ax.axhline(y=25, color="#C44E52", linestyle="--", linewidth=1.5, alpha=0.7, label="Overpriced threshold (+25%)")
ax.axhline(y=-25, color="#55A868", linestyle="--", linewidth=1.5, alpha=0.7, label="Deal threshold (-25%)")
ax.set_title("📊 Deal Detector: Which Phones Are Overpriced or Underpriced?", fontweight="bold", fontsize=16)
ax.set_xlabel("Actual Price (USD)")
ax.set_ylabel("Price Difference (%)\n(Positive = Overpriced, Negative = Good Deal)")
ax.legend(fontsize=11)
plt.colorbar(scatter, ax=ax, label="Price Difference (%)", shrink=0.8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "10_deal_detector.png"))
plt.close()
print("\n✅ Saved: 10_deal_detector.png")



📦 Training set: 663 samples
📦 Test set:     166 samples
📦 Features:     ['Weight_g', 'RAM_GB', 'Front_Camera_MP', 'Back_Camera_MP', 'Back_Camera_Count', 'Battery_mAh', 'Screen_inches', 'Year', 'Company_Encoded']

🔢 Polynomial Features: 9 → 54 features (degree=2)

----------------------------------------------------------------------
Model                                 MAE       RMSE         R²  CV R² (5-fold)
----------------------------------------------------------------------
Linear Regression              $      193 $      238    0.5987   0.6106±0.0533
Ridge Regression (α=1.0)       $      193 $      238    0.5985   0.6107±0.0531
Poly-2 Ridge Regression        $      137 $      177    0.7798   0.7379±0.0665
Random Forest Regressor        $       78 $      118    0.9022   0.8872±0.0389
Gradient Boosting              $       74 $      114    0.9079   0.8989±0.0451
HistGradientBoosting           $       82 $      124    0.8906   0.8540±0.0513
Extra Trees Regressor          $      

## PART 4: CLASSIFICATION — PRICE TIER PREDICTION

In [16]:
print("\n" + "=" * 70)

# --- 4.1 Create price tiers --------------------------------------------------
def assign_tier(price):
    if price < 300:
        return "Budget"
    elif price < 700:
        return "Mid-Range"
    elif price < 1000:
        return "Premium"
    else:
        return "Ultra-Premium"

df_clean["Price_Tier"] = df_clean["Price_USD"].apply(assign_tier)
tier_order = ["Budget", "Mid-Range", "Premium", "Ultra-Premium"]

print("\n📊 Price Tier Distribution:")
tier_dist = df_clean["Price_Tier"].value_counts()
for tier in tier_order:
    if tier in tier_dist:
        print(f"  {tier:<15}: {tier_dist[tier]:>4} phones ({tier_dist[tier]/len(df_clean)*100:.1f}%)")

# Encode tiers
le_tier = LabelEncoder()
le_tier.fit(tier_order)
df_clean["Price_Tier_Encoded"] = le_tier.transform(df_clean["Price_Tier"])

# --- 4.2 Prepare classification data ----------------------------------------
X_cls = df_clean[regression_features].values
y_cls = df_clean["Price_Tier_Encoded"].values

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=RANDOM_STATE, stratify=y_cls
)

scaler_cls = StandardScaler()
X_train_c_scaled = scaler_cls.fit_transform(X_train_c)
X_test_c_scaled = scaler_cls.transform(X_test_c)

# --- 4.3 Train classifiers ---------------------------------------------------
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

cls_results = {}
print("\n" + "-" * 70)
print(f"{'Model':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 70)

for name, clf in classifiers.items():
    if "Forest" in name or "Tree" in name:
        clf.fit(X_train_c, y_train_c)
        y_pred_c = clf.predict(X_test_c)
    else:
        clf.fit(X_train_c_scaled, y_train_c)
        y_pred_c = clf.predict(X_test_c_scaled)

    acc = accuracy_score(y_test_c, y_pred_c)
    prec = precision_score(y_test_c, y_pred_c, average="weighted", zero_division=0)
    rec = recall_score(y_test_c, y_pred_c, average="weighted", zero_division=0)
    f1 = f1_score(y_test_c, y_pred_c, average="weighted", zero_division=0)

    cls_results[name] = {
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1,
        "predictions": y_pred_c
    }
    print(f"{name:<25} {acc:>9.4f} {prec:>10.4f} {rec:>9.4f} {f1:>9.4f}")

print("-" * 70)

# --- 4.4 Confusion Matrix for best classifier --------------------------------
best_cls_name = max(cls_results, key=lambda k: cls_results[k]["Accuracy"])
best_cls_preds = cls_results[best_cls_name]["predictions"]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Confusion matrix
cm = confusion_matrix(y_test_c, best_cls_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=tier_order, yticklabels=tier_order,
            cbar_kws={"label": "Count"})
axes[0].set_title(f"Confusion Matrix — {best_cls_name}", fontweight="bold")
axes[0].set_xlabel("Predicted Tier")
axes[0].set_ylabel("Actual Tier")

# Classifier comparison
clf_names = list(cls_results.keys())
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1"]
x_pos = np.arange(len(clf_names))
width = 0.2
colors_cls = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

for i, metric in enumerate(metrics_to_plot):
    vals = [cls_results[n][metric] for n in clf_names]
    axes[1].bar(x_pos + i * width, vals, width, label=metric, color=colors_cls[i],
                edgecolor="white")

axes[1].set_xticks(x_pos + width * 1.5)
axes[1].set_xticklabels(clf_names, rotation=10)
axes[1].set_title("Classifier Performance Comparison", fontweight="bold")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1.15)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "11_classification_results.png"))
plt.close()
print(f"\n✅ Saved: 11_classification_results.png")
print(f"🏆 Best Classifier: {best_cls_name} (Accuracy = {cls_results[best_cls_name]['Accuracy']:.4f})")

# Print detailed classification report for best model
print(f"\n📋 Detailed Classification Report ({best_cls_name}):")
print(classification_report(y_test_c, best_cls_preds, target_names=tier_order, zero_division=0))

# --- 4.5 Price Tier Visualization ---------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Tier distribution pie chart
tier_counts = df_clean["Price_Tier"].value_counts().reindex(tier_order)
tier_colors = ["#55A868", "#4C72B0", "#DD8452", "#C44E52"]
wedges, texts, autotexts = axes[0].pie(
    tier_counts, labels=tier_order, autopct="%1.1f%%",
    colors=tier_colors, startangle=90, pctdistance=0.85,
    wedgeprops=dict(width=0.5, edgecolor="white", linewidth=2)
)
for t in autotexts:
    t.set_fontsize(12)
    t.set_fontweight("bold")
axes[0].set_title("Smartphone Price Tier Distribution", fontweight="bold", fontsize=15)

# Average specs per tier
tier_specs = df_clean.groupby("Price_Tier")[["RAM_GB", "Back_Camera_MP", "Battery_mAh"]].mean()
tier_specs = tier_specs.reindex(tier_order)
tier_specs_norm = tier_specs.div(tier_specs.max())  # Normalize for comparison

tier_specs_norm.plot(kind="bar", ax=axes[1], color=["#4C72B0", "#55A868", "#DD8452"],
                     edgecolor="white", width=0.7)
axes[1].set_title("Average Specs by Price Tier (Normalized)", fontweight="bold")
axes[1].set_xlabel("Price Tier")
axes[1].set_ylabel("Normalized Value")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(["RAM (GB)", "Camera (MP)", "Battery (mAh)"], fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "12_price_tiers.png"))
plt.close()
print("✅ Saved: 12_price_tiers.png")



📊 Price Tier Distribution:
  Budget         :  279 phones (33.7%)
  Mid-Range      :  293 phones (35.3%)
  Premium        :  129 phones (15.6%)
  Ultra-Premium  :  128 phones (15.4%)

----------------------------------------------------------------------
Model                       Accuracy  Precision     Recall         F1
----------------------------------------------------------------------
Logistic Regression          0.6566     0.6608    0.6566    0.6529
Decision Tree                0.7590     0.7619    0.7590    0.7568
Random Forest                0.7952     0.7926    0.7952    0.7924
----------------------------------------------------------------------

✅ Saved: 11_classification_results.png
🏆 Best Classifier: Random Forest (Accuracy = 0.7952)

📋 Detailed Classification Report (Random Forest):
               precision    recall  f1-score   support

       Budget       0.88      0.88      0.88        56
    Mid-Range       0.78      0.86      0.82        59
      Premium       

## PART 5: UNSUPERVISED ANALYSIS — PHONE CLUSTERING

**New techniques in this section:**
- Silhouette Score evaluation for cluster quality — Lab12
- DBSCAN clustering with `eps` hyperparameter search — Lab12
- GMM (Gaussian Mixture Model) clustering with `n_components` search — Lab12
- Clustering method comparison (KMeans vs DBSCAN vs GMM) — Lab12

In [17]:
print("\n" + "=" * 70)

# --- 5.1 K-Means Clustering --------------------------------------------------
cluster_features = ["RAM_GB", "Back_Camera_MP", "Battery_mAh", "Screen_inches", "Weight_g", "Price_USD"]
X_cluster = df_clean[cluster_features].values

scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

# Elbow method to find optimal K
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_cluster_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(list(K_range), inertias, "o-", color="#4C72B0", linewidth=2.5, markersize=10)
ax.set_title("Elbow Method — Finding Optimal Number of Clusters", fontweight="bold")
ax.set_xlabel("Number of Clusters (K)")
ax.set_ylabel("Inertia (Within-cluster Sum of Squares)")
ax.grid(True, alpha=0.3)
# Highlight the elbow (K=4)
ax.axvline(x=4, color="#C44E52", linestyle="--", linewidth=2, alpha=0.7, label="Suggested K = 4")
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "13_elbow_method.png"))
plt.close()
print("✅ Saved: 13_elbow_method.png")

# --- 5.2 Apply K-Means with K=4 ----------------------------------------------
kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
df_clean["Cluster"] = kmeans.fit_predict(X_cluster_scaled)

# --- 5.3 PCA for 2D visualization --------------------------------------------
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cluster_scaled)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Clusters in PCA space
cluster_colors = ["#4C72B0", "#55A868", "#C44E52", "#DD8452"]
for c in range(4):
    mask = df_clean["Cluster"] == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=cluster_colors[c],
                    label=f"Cluster {c}", alpha=0.6, edgecolors="white", s=60)
axes[0].set_title("Phone Clusters (PCA Projection)", fontweight="bold")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Actual price tiers in PCA space
tier_color_map = {"Budget": "#55A868", "Mid-Range": "#4C72B0",
                  "Premium": "#DD8452", "Ultra-Premium": "#C44E52"}
for tier in tier_order:
    mask = df_clean["Price_Tier"] == tier
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=tier_color_map[tier],
                    label=tier, alpha=0.6, edgecolors="white", s=60)
axes[1].set_title("Actual Price Tiers (PCA Projection)", fontweight="bold")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

fig.suptitle("K-Means Clusters vs. Actual Price Tiers", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "14_clustering_results.png"))
plt.close()
print("✅ Saved: 14_clustering_results.png")

# Cluster profiles
print("\n📊 Cluster Profiles (Average Values):")
cluster_profile = df_clean.groupby("Cluster")[cluster_features].mean().round(1)
print(cluster_profile.to_string())

# Cross-tabulation: Cluster vs. Price Tier
print("\n📊 Cluster vs. Price Tier Cross-tabulation:")
cross_tab = pd.crosstab(df_clean["Cluster"], df_clean["Price_Tier"],
                        margins=True, margins_name="Total")
cross_tab = cross_tab[tier_order + ["Total"]]
print(cross_tab.to_string())

# --- 5.4 Silhouette Score for KMeans (Lab12 technique) -----------------------
kmeans_silhouette = silhouette_score(X_cluster_scaled, df_clean["Cluster"])
print(f"\n📊 KMeans Silhouette Score (K=4): {kmeans_silhouette:.4f}")

# Silhouette score for different K values
print("\n📊 Silhouette Score by K (hyperparameter search):")
silhouette_scores_k = []
for k in range(2, 11):
    km_temp = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels_temp = km_temp.fit_predict(X_cluster_scaled)
    sil = silhouette_score(X_cluster_scaled, labels_temp)
    silhouette_scores_k.append(sil)
    print(f"  K={k}: Silhouette = {sil:.4f}")

best_k = list(range(2, 11))[np.argmax(silhouette_scores_k)]
print(f"  → Best K by Silhouette Score: {best_k} (score = {max(silhouette_scores_k):.4f})")

# --- 5.5 DBSCAN Clustering (Lab12 technique) ---------------------------------
print("\n--- DBSCAN Clustering ---")

# Hyperparameter search for eps (Lab12 technique: grid search with silhouette)
eps_range = [i / 10 for i in range(3, 15)]
dbscan_results = []
for eps_val in eps_range:
    db_model = DBSCAN(eps=eps_val, min_samples=5)
    db_labels = db_model.fit_predict(X_cluster_scaled)
    n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise = (db_labels == -1).sum()
    if n_clusters > 1:
        sil = silhouette_score(X_cluster_scaled, db_labels)
    else:
        sil = float('nan')
    dbscan_results.append({'eps': eps_val, 'n_clusters': n_clusters,
                           'n_noise': n_noise, 'silhouette': sil})

dbscan_df = pd.DataFrame(dbscan_results)
print(dbscan_df.to_string(index=False))

# Choose best eps by silhouette score
best_eps_row = dbscan_df.loc[dbscan_df['silhouette'].idxmax()]
best_eps = best_eps_row['eps']
print(f"\n→ Best DBSCAN eps = {best_eps} (Silhouette = {best_eps_row['silhouette']:.4f})")

# Apply best DBSCAN
dbscan_best = DBSCAN(eps=best_eps, min_samples=5)
df_clean["DBSCAN_Cluster"] = dbscan_best.fit_predict(X_cluster_scaled)
dbscan_n_clusters = len(set(df_clean["DBSCAN_Cluster"])) - (1 if -1 in df_clean["DBSCAN_Cluster"].values else 0)
dbscan_n_noise = (df_clean["DBSCAN_Cluster"] == -1).sum()
print(f"   DBSCAN found {dbscan_n_clusters} clusters + {dbscan_n_noise} noise points")

# --- 5.6 GMM Clustering (Lab12 technique) ------------------------------------
print("\n--- GMM (Gaussian Mixture Model) Clustering ---")

# Hyperparameter search for n_components
gmm_results = []
for n_comp in range(2, 11):
    gmm_model = GaussianMixture(n_components=n_comp, random_state=RANDOM_STATE)
    gmm_labels = gmm_model.fit_predict(X_cluster_scaled)
    sil = silhouette_score(X_cluster_scaled, gmm_labels)
    gmm_results.append({'n_components': n_comp, 'silhouette': sil, 'BIC': gmm_model.bic(X_cluster_scaled)})
    print(f"  n_components={n_comp}: Silhouette = {sil:.4f}, BIC = {gmm_model.bic(X_cluster_scaled):,.0f}")

gmm_df = pd.DataFrame(gmm_results)
best_gmm_row = gmm_df.loc[gmm_df['silhouette'].idxmax()]
best_n_comp = int(best_gmm_row['n_components'])
print(f"\n→ Best GMM n_components = {best_n_comp} (Silhouette = {best_gmm_row['silhouette']:.4f})")

# Apply best GMM
gmm_best = GaussianMixture(n_components=best_n_comp, random_state=RANDOM_STATE)
df_clean["GMM_Cluster"] = gmm_best.fit_predict(X_cluster_scaled)

# --- 5.7 Clustering Comparison (KMeans vs DBSCAN vs GMM) ----------------------
print("\n📊 Clustering Method Comparison:")
print("-" * 55)

comparison_data = {
    "KMeans (K=4)": {
        "Silhouette": kmeans_silhouette,
        "N_Clusters": 4,
    },
}

# DBSCAN silhouette (only if valid clusters exist)
dbscan_valid = df_clean["DBSCAN_Cluster"] != -1
if dbscan_valid.sum() > 0 and dbscan_n_clusters > 1:
    dbscan_sil = silhouette_score(
        X_cluster_scaled[dbscan_valid],
        df_clean.loc[dbscan_valid, "DBSCAN_Cluster"]
    )
    comparison_data[f"DBSCAN (eps={best_eps})"] = {
        "Silhouette": dbscan_sil,
        "N_Clusters": dbscan_n_clusters,
    }

gmm_sil = silhouette_score(X_cluster_scaled, df_clean["GMM_Cluster"])
comparison_data[f"GMM (n={best_n_comp})"] = {
    "Silhouette": gmm_sil,
    "N_Clusters": best_n_comp,
}

print(f"{'Method':<25} {'Silhouette':>12} {'Clusters':>10}")
print("-" * 55)
for method, vals in comparison_data.items():
    print(f"{method:<25} {vals['Silhouette']:>12.4f} {vals['N_Clusters']:>10}")
print("-" * 55)

# --- 5.8 Visualize Clustering Comparison --------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# KMeans
for c in range(4):
    mask = df_clean["Cluster"] == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.6, edgecolors="white", s=50,
                    label=f"Cluster {c}")
axes[0].set_title(f"KMeans (K=4)\nSilhouette = {kmeans_silhouette:.4f}", fontweight="bold")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# DBSCAN
db_labels_unique = sorted(df_clean["DBSCAN_Cluster"].unique())
for c in db_labels_unique:
    mask = df_clean["DBSCAN_Cluster"] == c
    label = f"Noise" if c == -1 else f"Cluster {c}"
    color = "gray" if c == -1 else None
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.6 if c != -1 else 0.3,
                    edgecolors="white", s=50, label=label, c=color)
axes[1].set_title(f"DBSCAN (eps={best_eps})\n{dbscan_n_clusters} clusters, {dbscan_n_noise} noise",
                  fontweight="bold")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

# GMM
for c in range(best_n_comp):
    mask = df_clean["GMM_Cluster"] == c
    axes[2].scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.6, edgecolors="white", s=50,
                    label=f"Cluster {c}")
axes[2].set_title(f"GMM (n={best_n_comp})\nSilhouette = {gmm_sil:.4f}", fontweight="bold")
axes[2].set_xlabel("PC1"); axes[2].set_ylabel("PC2")
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)

fig.suptitle("Clustering Method Comparison (PCA Projection)", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "19_clustering_comparison.png"))
plt.close()
print("✅ Saved: 19_clustering_comparison.png")

# --- 5.9 Silhouette Score Comparison Plot ------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# KMeans silhouette by K
axes[0].plot(range(2, 11), silhouette_scores_k, "o-", color="#4C72B0", linewidth=2.5, markersize=8)
axes[0].axvline(x=best_k, color="#C44E52", linestyle="--", linewidth=2, alpha=0.7,
                label=f"Best K = {best_k}")
axes[0].set_title("KMeans: Silhouette Score by K", fontweight="bold")
axes[0].set_xlabel("Number of Clusters (K)")
axes[0].set_ylabel("Silhouette Score")
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Method comparison bar chart
methods = list(comparison_data.keys())
sil_vals = [comparison_data[m]["Silhouette"] for m in methods]
bar_colors = ["#4C72B0", "#55A868", "#DD8452"][:len(methods)]
bars = axes[1].bar(range(len(methods)), sil_vals, color=bar_colors, edgecolor="white", width=0.5)
axes[1].set_xticks(range(len(methods)))
axes[1].set_xticklabels(methods, rotation=15, ha="right")
axes[1].set_title("Clustering Method Comparison — Silhouette Score", fontweight="bold")
axes[1].set_ylabel("Silhouette Score")
for bar, score in zip(bars, sil_vals):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{score:.4f}", ha="center", fontweight="bold", fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "20_silhouette_comparison.png"))
plt.close()
print("✅ Saved: 20_silhouette_comparison.png")


✅ Saved: 13_elbow_method.png
✅ Saved: 14_clustering_results.png

📊 Cluster Profiles (Average Values):
         RAM_GB  Back_Camera_MP  Battery_mAh  Screen_inches  Weight_g  Price_USD
Cluster                                                                         
0          12.5            86.8       4913.8            7.4     241.0     1714.3
1          10.7            58.0       4765.9            6.7     200.9      837.2
2           5.1            18.7       3292.0            6.0     171.7      671.3
3           6.4            50.2       4890.8            6.6     189.7      318.5

📊 Cluster vs. Price Tier Cross-tabulation:
Price_Tier  Budget  Mid-Range  Premium  Ultra-Premium  Total
Cluster                                                     
0                0          0        0             38     38
1                7         93       86             78    264
2               26         19       40             12     97
3              246        181        3              0    430
T

## PART 6: CROSS-COUNTRY PRICE ANALYSIS

In [18]:
print("\n" + "=" * 70)

# --- 6.1 Convert prices to USD for comparison --------------------------------
# Approximate exchange rates (as of 2024-2025)
exchange_rates = {
    "Price_PKR": 278.0,   # PKR to USD
    "Price_INR": 83.5,    # INR to USD
    "Price_CNY": 7.25,    # CNY to USD
    "Price_USD": 1.0,     # Already USD
    "Price_AED": 3.67,    # AED to USD
}

country_names = {
    "Price_PKR": "Pakistan",
    "Price_INR": "India",
    "Price_CNY": "China",
    "Price_USD": "USA",
    "Price_AED": "Dubai (UAE)",
}

for col, rate in exchange_rates.items():
    df_clean[f"{col}_USD_equiv"] = df_clean[col] / rate

# --- 6.2 Price comparison across countries ------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Average price comparison
usd_equiv_cols = [f"{col}_USD_equiv" for col in exchange_rates.keys()]
avg_prices_by_country = df_clean[usd_equiv_cols].mean()
country_labels = [country_names[col.replace("_USD_equiv", "")] for col in usd_equiv_cols]

bars = axes[0].bar(country_labels, avg_prices_by_country.values,
                   color=sns.color_palette("Set2", 5), edgecolor="white", width=0.6)
axes[0].set_title("Average Smartphone Price by Country (in USD)", fontweight="bold")
axes[0].set_ylabel("Price (USD equivalent)")
for bar, val in zip(bars, avg_prices_by_country.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                 f"${val:,.0f}", ha="center", fontweight="bold", fontsize=11)
axes[0].tick_params(axis="x", rotation=15)

# Brand markup by country
top_5_brands = df_clean["Company"].value_counts().head(5).index
brand_country_data = []
for brand in top_5_brands:
    brand_data = df_clean[df_clean["Company"] == brand]
    for col in exchange_rates.keys():
        avg = brand_data[f"{col}_USD_equiv"].mean()
        brand_country_data.append({
            "Brand": brand,
            "Country": country_names[col],
            "Avg_Price_USD": avg
        })

df_brand_country = pd.DataFrame(brand_country_data)
pivot = df_brand_country.pivot(index="Brand", columns="Country", values="Avg_Price_USD")
pivot.plot(kind="bar", ax=axes[1], width=0.7, edgecolor="white")
axes[1].set_title("Average Price by Brand & Country (Top 5 Brands)", fontweight="bold")
axes[1].set_ylabel("Price (USD equivalent)")
axes[1].tick_params(axis="x", rotation=15)
axes[1].legend(fontsize=9, loc="upper left")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "15_cross_country_prices.png"))
plt.close()
print("✅ Saved: 15_cross_country_prices.png")

# --- 6.3 Which country is cheapest for each brand? ---------------------------
print("\n📊 Cheapest Country for Each Top Brand:")
for brand in top_5_brands:
    brand_data = df_clean[df_clean["Company"] == brand]
    avg_by_country = {}
    for col, name in country_names.items():
        avg_by_country[name] = brand_data[f"{col}_USD_equiv"].mean()
    cheapest = min(avg_by_country, key=avg_by_country.get)
    cheapest_price = avg_by_country[cheapest]
    most_exp = max(avg_by_country, key=avg_by_country.get)
    most_exp_price = avg_by_country[most_exp]
    diff_pct = ((most_exp_price - cheapest_price) / cheapest_price) * 100
    print(f"  {brand:<12}: Cheapest in {cheapest:<15} (${cheapest_price:,.0f}) | "
          f"Most expensive in {most_exp:<15} (${most_exp_price:,.0f}) | "
          f"Diff: {diff_pct:.1f}%")

# --- 6.4 Price markup heatmap by brand and country ----------------------------
fig, ax = plt.subplots(figsize=(12, 8))

# Calculate price premium relative to USA price
markup_data = []
for brand in top_5_brands:
    brand_data = df_clean[df_clean["Company"] == brand]
    usa_avg = brand_data["Price_USD"].mean()
    for col, name in country_names.items():
        if col == "Price_USD":
            continue
        equiv_avg = brand_data[f"{col}_USD_equiv"].mean()
        markup_pct = ((equiv_avg - usa_avg) / usa_avg) * 100
        markup_data.append({"Brand": brand, "Country": name, "Markup_%": markup_pct})

df_markup = pd.DataFrame(markup_data)
markup_pivot = df_markup.pivot(index="Brand", columns="Country", values="Markup_%")

sns.heatmap(markup_pivot, annot=True, fmt=".1f", cmap="RdYlGn_r", center=0,
            ax=ax, linewidths=1, cbar_kws={"label": "Price Markup vs USA (%)"})
ax.set_title("Price Markup vs. USA by Brand & Country (%)", fontweight="bold", fontsize=16)
ax.set_xlabel("Country")
ax.set_ylabel("Brand")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "16_country_markup_heatmap.png"))
plt.close()
print("✅ Saved: 16_country_markup_heatmap.png")


✅ Saved: 15_cross_country_prices.png

📊 Cheapest Country for Each Top Brand:
  Oppo        : Cheapest in Pakistan        ($339) | Most expensive in Dubai (UAE)     ($531) | Diff: 56.3%
  Apple       : Cheapest in Pakistan        ($933) | Most expensive in India           ($1,245) | Diff: 33.4%
  Vivo        : Cheapest in Pakistan        ($260) | Most expensive in USA             ($476) | Diff: 82.9%
  Samsung     : Cheapest in China           ($730) | Most expensive in Dubai (UAE)     ($824) | Diff: 12.9%
  Honor       : Cheapest in Pakistan        ($472) | Most expensive in Dubai (UAE)     ($676) | Diff: 43.2%
✅ Saved: 16_country_markup_heatmap.png


## FINAL SUMMARY

In [19]:
print("\n" + "=" * 70)
print("📝 PROJECT SUMMARY")
print(f"""
Dataset: Mobiles Dataset 2025 ({df_clean.shape[0]} phones, {df_clean.shape[1]} features)
Target: Smartphone Price (USD)

📊 EDA Highlights:
  • {df_clean['Company'].nunique()} brands, years {df_clean['Year'].min()}-{df_clean['Year'].max()}
  • Median price: ${df_clean['Price_USD'].median():,.0f} | Mean price: ${df_clean['Price_USD'].mean():,.0f}
  • Most represented brand: {df_clean['Company'].value_counts().index[0]} ({df_clean['Company'].value_counts().values[0]} models)
  • Pearson correlation significance testing: All features tested with p-values

🔮 Regression (Best: {best_model_name}):
  • R² = {results[best_model_name]['R²']:.4f}
  • CV R² (5-fold) = {results[best_model_name]['CV_R2_mean']:.4f} ± {results[best_model_name]['CV_R2_std']:.4f}
  • MAE = ${results[best_model_name]['MAE']:,.0f}
  • RMSE = ${results[best_model_name]['RMSE']:,.0f}
  • Models: {len(models)} total (including HistGradientBoosting, Poly-2 Ridge)

🏷️ Classification (Best: {best_cls_name}):
  • Accuracy = {cls_results[best_cls_name]['Accuracy']:.4f}
  • F1 Score = {cls_results[best_cls_name]['F1']:.4f}

🔍 Key Feature Drivers (by importance):
  1. {feat_importance.index[-1]} ({feat_importance.values[-1]:.3f})
  2. {feat_importance.index[-2]} ({feat_importance.values[-2]:.3f})
  3. {feat_importance.index[-3]} ({feat_importance.values[-3]:.3f})

📦 Clustering Comparison:
  • KMeans (K=4): Silhouette = {kmeans_silhouette:.4f}
  • DBSCAN (eps={best_eps}): {dbscan_n_clusters} clusters + {dbscan_n_noise} noise
  • GMM (n={best_n_comp}): Silhouette = {gmm_sil:.4f}

🧪 Course Techniques Applied:
  • Feature binning (pd.cut, pd.qcut) — Lab05
  • Polynomial Features (degree=2) — Lab06
  • HistGradientBoostingRegressor — Lab07/10
  • Cross-validation (5-fold CV) — Lab10
  • Learning Curves (overfitting diagnosis) — Lab10
  • Partial Dependency Plots — Lab10
  • Pearson Correlation with p-values — Lab10
  • Model export with pickle — Lab10
  • DBSCAN clustering — Lab12
  • GMM (Gaussian Mixture) clustering — Lab12
  • Silhouette Score evaluation — Lab12
  • Hyperparameter grid search — Lab12

📁 All plots saved to: {OUTPUT_DIR}/ (20 total)
📁 Best model exported to: best_model.pkl
""")

print("✅ Project execution complete!")


📝 PROJECT SUMMARY

Dataset: Mobiles Dataset 2025 (829 phones, 53 features)
Target: Smartphone Price (USD)

📊 EDA Highlights:
  • 17 brands, years 2014-2025
  • Median price: $449 | Mean price: $589
  • Most represented brand: Oppo (123 models)
  • Pearson correlation significance testing: All features tested with p-values

🔮 Regression (Best: Extra Trees Regressor):
  • R² = 0.9097
  • CV R² (5-fold) = 0.8909 ± 0.0524
  • MAE = $74
  • RMSE = $113
  • Models: 8 total (including HistGradientBoosting, Poly-2 Ridge)

🏷️ Classification (Best: Random Forest):
  • Accuracy = 0.7952
  • F1 Score = 0.7924

🔍 Key Feature Drivers (by importance):
  1. Weight_g (0.373)
  2. Screen_inches (0.184)
  3. RAM_GB (0.150)

📦 Clustering Comparison:
  • KMeans (K=4): Silhouette = 0.2830
  • DBSCAN (eps=1.4): 5 clusters + 22 noise
  • GMM (n=2): Silhouette = 0.2397

🧪 Course Techniques Applied:
  • Feature binning (pd.cut, pd.qcut) — Lab05
  • Polynomial Features (degree=2) — Lab06
  • HistGradientBoostin